# 01 — Treino: baseline LIBERO (1 tarefa, sem linguagem)

Notebook fino: toda a lógica vive em `src/act_lang/`. Aqui só orquestração e visualização.

In [ ]:
# --- Setup: clonar (limpo) + instalar em modo editável ---
# Repo privado? Guarde um token no Secrets do Colab e use:
#   from google.colab import userdata; token = userdata.get("GH_TOKEN")
#   !git clone https://{token}@github.com/rafaelheydt/act-lang.git /content/act-lang
#
# %cd /content ANTES do rm -rf: se uma execução anterior desta célula deixou
# o shell dentro de /content/act-lang, apagar essa pasta com o shell "sentado"
# nela quebra o cwd do processo (erros "getcwd: cannot access parent
# directories") e derruba até o git clone seguinte.
%cd /content
!rm -rf /content/act-lang
!git clone https://github.com/rafaelheydt/act-lang.git /content/act-lang
%cd /content/act-lang
!pip install -q -e . "lerobot[libero]"

import sys
# "configs/" fica FORA de src/ de propósito (configs de experimento editáveis
# sem reinstalar nada) -- por isso não é abrangido pelo pip install -e .
# Alguns kernels IPython não resolvem import a partir do cwd dinamicamente,
# então o insert explícito é a forma confiável de tornar "from configs...."
# importável, independente de como aquele kernel específico se comporta.
if "/content/act-lang" not in sys.path:
    sys.path.insert(0, "/content/act-lang")

import os
os.environ["MUJOCO_GL"] = "egl"  # headless (Colab)

# SEM autoreload: o IPython pré-instalado no Colab (pinado em 7.34.0 pelo
# pacote google-colab) quebra com o autoreload no runtime atual, e forçar o
# upgrade do IPython quebra drive.mount()/exibição de vídeo em troca.
#
# IMPORTANTE: depois desta célula rodar pela primeira vez em CADA runtime
# novo, faça Runtime > Restart session antes de continuar -- o Python só lê
# o registro do "pip install -e ." (arquivo .pth) na inicialização do
# interpretador, não em tempo real. Sem o restart, "import act_lang" falha
# com ModuleNotFoundError mesmo com tudo instalado corretamente.
#
# Reload cirúrgico sem reiniciar, se preferir (depois do primeiro restart):
#   import importlib, act_lang.models.act
#   importlib.reload(act_lang.models.act)

In [ ]:
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

from configs.libero_single_task import CONFIG as cfg
from act_lang.data.libero import (
    REPO_ID, LiberoActBridge, filter_episodes_by_tasks,
    make_delta_timestamps, split_episodes,
)
from act_lang.data.normalize import MinMaxNormalizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
# --- Dados: filtro por tarefa, split por episódio, loaders ---
meta = LeRobotDatasetMetadata(REPO_ID)
full_dataset = LeRobotDataset(REPO_ID)

episode_ids = filter_episodes_by_tasks(meta, full_dataset, cfg["task_texts"])
train_ids, val_ids = split_episodes(episode_ids, cfg["val_frac"], cfg["seed"])
print(f"episódios: {len(episode_ids)} -> train {len(train_ids)} | val {len(val_ids)}")
# ATENÇÃO: com poucas dezenas de episódios, val é pequeno — métricas de
# validação são ruidosas; interprete o "melhor checkpoint" com essa lente.

delta_ts = make_delta_timestamps(meta.fps, cfg["obs_horizon"], cfg["pred_horizon"])
train_dataset = LeRobotDataset(REPO_ID, episodes=train_ids, delta_timestamps=delta_ts)
val_dataset = LeRobotDataset(REPO_ID, episodes=val_ids, delta_timestamps=delta_ts)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=cfg["batch_size"], shuffle=True,
    num_workers=0, drop_last=True,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=0,
)

state_norm = MinMaxNormalizer.from_lerobot_stats(meta.stats, "observation.state").to(device)
action_norm = MinMaxNormalizer.from_lerobot_stats(meta.stats, "action").to(device)
bridge = LiberoActBridge(state_norm, action_norm)

In [ ]:
# --- Modelo + optimizer ---
from act_lang.models.act import ACT
from act_lang.models.backbone import freeze_batchnorm
from act_lang.training.optim import build_optimizer

model = ACT(
    action_dim=cfg["action_dim"], state_dim=cfg["state_dim"],
    d_model=cfg["d_model"], latent_dim=cfg["latent_dim"],
    chunk_size=cfg["chunk_size"], n_cameras=cfg["n_cameras"],
    n_encoder_layers=cfg["n_encoder_layers"], n_decoder_layers=cfg["n_decoder_layers"],
    n_heads=cfg["n_heads"], dropout=cfg["dropout"], pretrained_backbone=True,
    decoder_style=cfg["decoder_style"],
)
if cfg["freeze_bn"]:
    freeze_batchnorm(model.vision_backbone)
model = model.to(device)
print(f"parâmetros: {sum(p.numel() for p in model.parameters()):,}")

optimizer = build_optimizer(model, cfg["lr"], cfg["lr_backbone"], cfg["weight_decay"])

In [ ]:
# --- Smoke test: um batch real, forward + backward ---
from act_lang.training.loss import act_loss

batch = next(iter(train_loader))
images, state, actions, is_pad, task_texts = bridge(batch, device)
print(f"images {tuple(images.shape)} | state {tuple(state.shape)} | actions {tuple(actions.shape)}")

pred, mu, logvar = model(images, state, actions=actions, is_pad=is_pad)
loss, recon, kld = act_loss(pred, actions, mu, logvar, is_pad, cfg["kl_weight"], cfg["free_bits"])
loss.backward(); optimizer.zero_grad()
print(f"loss {loss.item():.4f} | recon {recon.item():.4f} | kld {kld.item():.4f}")

In [ ]:
# --- Drive + treino completo ---
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

checkpoint_dir = Path("/content/drive/MyDrive") / cfg["experiment_name"]

from act_lang.training.loop import fit

# retomar de sessão caída:
#   from act_lang.training.checkpoints import load_checkpoint
#   start_epoch, history = load_checkpoint(
#       checkpoint_dir / "last_checkpoint.pt", model, optimizer, device)
#   e passe start_epoch=start_epoch, history=history ao fit(...)

history = fit(
    model, train_loader, val_loader, bridge, optimizer, device,
    checkpoint_dir=checkpoint_dir, num_epochs=cfg["num_epochs"],
    kl_weight=cfg["kl_weight"], free_bits=cfg["free_bits"],
    grad_clip_norm=cfg["grad_clip_norm"], patience=cfg["patience"],
    checkpoint_every=cfg["checkpoint_every"],
)

In [ ]:
# --- Curvas ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss total (recon + kl_weight*KL)")

axes[1].plot(history["train_recon"], label="train (z~q)")
axes[1].plot(history["val_recon"], label="val (z=mu)")
axes[1].plot(history["val_recon_z0"], label="val (z=0)", linestyle="--")
axes[1].set_title("Recon L1 — z0 é a métrica de seleção")

axes[2].plot(history["train_kld"], label="train")
axes[2].plot(history["val_kld"], label="val")
axes[2].set_title("kld_raw")

for ax in axes:
    ax.set_xlabel("época"); ax.legend()
plt.tight_layout(); plt.show()